Установим зависимости

In [ ]:
!pip install datasets evaluate seqeval -q

## Загрузка датасета

In [2]:
from datasets import load_dataset
import numpy as np
import torch
import os

raw_datasets = load_dataset("conll2003", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

conll2003.py:   0%|          | 0.00/9.57k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [3]:
from transformers import AutoTokenizer

def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            # Start of a new word!
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            # Special token
            new_labels.append(-100)
        else:
            # Same word as previous token
            label = labels[word_id]
            # If the label is B-XXX we change it to I-XXX
            if label % 2 == 1:
                label += 1
            new_labels.append(label)

    return new_labels

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs


model_checkpoint = "dslim/bert-large-NER"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

ner_feature = raw_datasets["train"].features["ner_tags"]

label_names = ner_feature.feature.names

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [5]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=8
)

In [6]:
id2label = {
    0: "O",
    1: "B-MISC",
    2: "I-MISC",
    3: "B-PER",
    4: "I-PER",
    5: "B-ORG",
    6: "I-ORG",
    7: "B-LOC",
    8: "I-LOC"
}

def postprocess(predictions, labels):
    predictions = predictions.detach().cpu().clone().numpy()
    labels = labels.detach().cpu().clone().numpy()

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return true_labels, true_predictions

Загружаем модель

In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    device_map='cuda'
)

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Кластеризируем веса

In [8]:
from sklearn.cluster import KMeans
from tqdm import tqdm

NUM_CLUSTERS = 128

model.to('cpu')

def weight_clustering(model):
    with torch.no_grad():
        for name, params in tqdm(model.named_parameters()):
            if not name.startswith('bert.embeddings'): # do not cluster embedding layer
                param_shape=list(params.size())
                weights=torch.flatten(params)
                weights=params.reshape(-1,1)
                kmeans = KMeans(n_clusters=min(NUM_CLUSTERS, len(weights)), random_state=0).fit(weights)

                cluster_centers = kmeans.cluster_centers_
                new_weights = torch.tensor([cluster_centers[i].item() for i in kmeans.labels_], dtype=weights.dtype)

                reshape_size_tuple=tuple(param_shape)
                new_weights=new_weights.reshape(reshape_size_tuple)
                params.data=new_weights.data

In [9]:
weight_clustering(model)

391it [1:12:10, 11.08s/it]


посмотрим на кол-во уникальных весов после кластеризации

In [11]:
len(next(model.bert.encoder.parameters()).unique())

128

## Замер метрик и скорости

In [12]:
import evaluate
import torch
from tqdm import tqdm

def evaluate_bert_ner(eval_dataloader, model, device):
    metric = evaluate.load("seqeval")
    for batch in tqdm(eval_dataloader):
        with torch.no_grad():
            batch = {k: v.to(device)for k, v in batch.items()}
            outputs = model(**batch)

        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]

        true_predictions, true_labels = postprocess(predictions, labels)
        metric.add_batch(predictions=true_predictions, references=true_labels)

    results = metric.compute()
    return results['overall_f1']


from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import time
from tqdm import tqdm


# Среднее время работы
def measure_time(nlp, raw_datasets, index, n):
    tsum = 0
    for i in tqdm(range(n)):
        start = time.perf_counter()
        ner_results = nlp(' '.join(raw_datasets["test"][index]["tokens"]))
        tsum += (time.perf_counter() - start)
    return tsum / n

Замерим метрики для начала

In [13]:
model.to('cuda:0')

f1_metric = evaluate_bert_ner(eval_dataloader, model, 'cuda:0')
print(f1_metric)

100%|██████████| 407/407 [00:30<00:00, 13.25it/s]


0.8296320051310833


Замеряем время работы

In [14]:
nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use cuda


In [18]:
mean_time_cuda = measure_time(nlp, raw_datasets, 3, 100)
print()
print('Mean time on cuda:', mean_time_cuda*1000, "ms")

100%|██████████| 100/100 [00:01<00:00, 57.44it/s]


Mean time on cuda: 17.297241260012015 ms


Проверим вес модели после GZIP

In [22]:
from transformers import AutoModelForTokenClassification

original_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    device_map='cuda'
)

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [24]:
torch.save(model.state_dict(), "clustered_model.pt")
torch.save(original_model.state_dict(), "original_model.pt")

In [27]:
!ls -lh

total 2.5G
-rw-r--r-- 1 root root 1.3G Apr 13 21:09 clustered_model.pt
-rw-r--r-- 1 root root 1.3G Apr 13 21:09 original_model.pt
drwxr-xr-x 1 root root 4.0K Apr 10 13:37 sample_data


In [31]:
!gzip -k clustered_model.pt original_model.pt

In [32]:
!ls -lh

total 4.1G
-rw-r--r-- 1 root root 1.3G Apr 13 21:09 clustered_model.pt
-rw-r--r-- 1 root root 464M Apr 13 21:09 clustered_model.pt.gz
-rw-r--r-- 1 root root 1.3G Apr 13 21:09 original_model.pt
-rw-r--r-- 1 root root 1.2G Apr 13 21:09 original_model.pt.gz
drwxr-xr-x 1 root root 4.0K Apr 10 13:37 sample_data


Видим, что кластеризация успешно работает! Размер модели уменьшился в 3 раза